# 05 — Run the Active Learning Loop

Run the full autonomous Active Learning pipeline.
The `ActiveLearningLoop` class orchestrates the Designer, Lab, and Learner over `max_cycles` iterations.

**Output:** `workspace/calibrated_parts.pkl`

In [ ]:
import os
# Navigate to repo root so all relative paths resolve correctly.
if os.path.basename(os.getcwd()) in ('notebooks', 'examples'):
    os.chdir('..')

In [ ]:
import pickle
import numpy as np
import warnings
warnings.filterwarnings("ignore")
from active_learning.config import ActiveLearningConfig
from active_learning.loop import ActiveLearningLoop
from gcad.circuit import Topo  # required for pickle deserialization

## Load Data

In [ ]:
# Load circuits
with open(os.path.join("data", "selected_M_circuits.pkl"), "rb") as f:
    circuits_list = pickle.load(f)
circuit_dict = {f"Circuit_{i+1}": c for i, c in enumerate(circuits_list)}

# Load hidden ground truth (defined in notebook 02)
with open(os.path.join("data", "ground_truth", "true_parts.pkl"), "rb") as f:
    true_parts = pickle.load(f)

print(f"Loaded {len(circuit_dict)} circuits.")

## Configure and Run

In [ ]:
config = ActiveLearningConfig(
    max_cycles=10,
    ensemble_size=200,
    spread_factor=2.0,
    dist_type='lognormal',
    budget_circuits=2,
    budget_dosages=2,
    dosages=list(np.round(np.arange(0.2, 4.2, 0.2), 2)),
    measurement_noise_std=5.0,
    perturbation_scale=0.1,
    selection_ratio=0.2
)

np.random.seed(42)
engine = ActiveLearningLoop(circuit_dict, true_parts, config)
calibrated_parts, history = engine.run()

## Convergence Plots

In [ ]:
import matplotlib.pyplot as plt

errors = [h['error'] for h in history]
variances = [h['variance_max'] for h in history]
cycles = list(range(len(history)))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(cycles, variances, marker='o', color='#1f77b4', linewidth=2)
axes[0].set_title('Designer Uncertainty', fontweight='bold')
axes[0].set_xlabel('AL Cycle')
axes[0].set_ylabel('Max Variance in Grid')

axes[1].plot(cycles, errors, marker='s', color='#d62728', linewidth=2)
axes[1].set_title('Learner Error (NMSE)', fontweight='bold')
axes[1].set_xlabel('AL Cycle')
axes[1].set_ylabel('Best NMSE')
axes[1].set_yscale('log')

plt.suptitle('Active Learning Convergence', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## Save Calibrated Parameters

In [ ]:
os.makedirs("workspace", exist_ok=True)
with open("workspace/calibrated_parts.pkl", "wb") as f:
    pickle.dump(calibrated_parts, f)
print("Saved calibrated parameters to 'workspace/calibrated_parts.pkl'.")

## Phenotypic Comparison — Hidden Truth vs. Calibrated Parameters

In [ ]:
import numpy as np
from scipy.integrate import odeint
from gcad.equations import system_equations_DsRed_pop

# Load promo params and nominal parts for building the discovered parts dict
with open(os.path.join("gcad", "data", "promo.pkl"), "rb") as f:
    promo_params = pickle.load(f)
with open(os.path.join("gcad", "data", "parts.pkl"), "rb") as f:
    discovered_parts = pickle.load(f)

# Overwrite with calibrated values
for k, v in calibrated_parts.items():
    discovered_parts[k] = v

t_span = config.get_t_span()
test_dosages = [0.2, 1.4, 4.0]
colors = ['#1f77b4', '#2ca02c', '#d62728']

n_circuits = len(circuit_dict)
fig, axes = plt.subplots(n_circuits, 2, figsize=(12, 4 * n_circuits), sharex=True, sharey='row')
if n_circuits == 1:
    axes = axes[None, :]

for row_idx, (c_name, topology) in enumerate(circuit_dict.items()):
    ax_truth = axes[row_idx, 0]
    ax_disc  = axes[row_idx, 1]

    for d_idx, dose in enumerate(test_dosages):
        sim_dose = {k: v * dose for k, v in topology.dose.items() if k != 'Rep'}
        if 'Rep' in topology.dose:
            sim_dose['Rep'] = topology.dose['Rep']
        temp_topo = Topo(topology.edge_list, sim_dose, topology.promo_node)

        y_truth = odeint(system_equations_DsRed_pop, np.zeros(temp_topo.num_states * 2), t_span,
                         args=('on', np.ones(5), temp_topo, promo_params, true_parts))[:, -1]
        y_disc  = odeint(system_equations_DsRed_pop, np.zeros(temp_topo.num_states * 2), t_span,
                         args=('on', np.ones(5), temp_topo, promo_params, discovered_parts))[:, -1]

        ax_truth.plot(t_span, y_truth, color=colors[d_idx], lw=2, label=f"{dose}x")
        ax_disc.plot(t_span, y_disc,  color=colors[d_idx], lw=2, linestyle='--')

    ax_truth.set_title(f"{c_name} — Hidden Truth", fontweight='bold')
    ax_disc.set_title(f"{c_name} — AL Calibrated", fontweight='bold')
    ax_truth.set_ylabel("Fluorescence")
    if row_idx == 0:
        ax_truth.legend(loc='upper left', fontsize=9)

axes[-1, 0].set_xlabel("Time (hrs)", fontweight='bold')
axes[-1, 1].set_xlabel("Time (hrs)", fontweight='bold')
plt.suptitle("Phenotypic Comparison: Hidden Truth vs. AL Calibrated", fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()